# Exercise 2: PCA on Beers Dataset

We study 40 beers each characterised by four variables:

| Variable | Meaning |
|----------|----------------------------------------------|
| Taste | Quantitative appreciation of taste by experts |
| Bitter | High = bitter, low = sweet |
| Thirst | High = thirst-quenching, low = still thirsty |
| DgAlcohol | Degree of alcohol |

We use PCA to summarise and visualise the data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Load data — first column is beer name, used as index
beers = pd.read_csv('../beers_data.csv', index_col=0)

# Round near-zero floating point noise to 0
beers = beers.round(10)

print(f'Shape: {beers.shape}')
beers.head()

In [ ]:
# Standardise (scale=TRUE equivalent) and run PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(beers)

pca = PCA(n_components=4)
scores = pca.fit_transform(X_scaled)   # individual coordinates (scores)

# Wrap scores in a DataFrame for convenience
score_df = pd.DataFrame(
    scores,
    columns=[f'PC{i+1}' for i in range(4)],
    index=beers.index
)

print('Explained variance ratio:', pca.explained_variance_ratio_.round(4))
print('Cumulative:', np.cumsum(pca.explained_variance_ratio_).round(4))

## Question 1: Individuals plot — first principal plane (PC1 vs PC2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(score_df['PC1'], score_df['PC2'], s=30, color='steelblue', zorder=3)

for beer, row in score_df.iterrows():
    ax.annotate(
        beer,
        xy=(row['PC1'], row['PC2']),
        fontsize=6.5,
        ha='left',
        va='bottom',
        xytext=(3, 3),
        textcoords='offset points'
    )

ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.7, linestyle='--')

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100
ax.set_xlabel(f'PC1 ({var1:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({var2:.1f}% variance)', fontsize=11)
ax.set_title('Individuals in the first principal plane', fontsize=13)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Question 2: Correlation circle — variables in the first principal plane

The correlation between original variable $j$ and principal component $k$ is:
$$r_{j,k} = \text{loading}_{j,k} \times \sqrt{\lambda_k}$$
where $\lambda_k$ is the $k$-th eigenvalue (explained variance). These coordinates are plotted inside a unit circle.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

# Correlation coordinates = loadings * sqrt(eigenvalue)
loadings = pca.components_.T  # shape (4 variables, 4 components)
eigenvalues = pca.explained_variance_
correlations = loadings * np.sqrt(eigenvalues)  # broadcast over rows

# Plot unit circle
theta = np.linspace(0, 2 * np.pi, 300)
ax.plot(np.cos(theta), np.sin(theta), 'k-', linewidth=0.8)

colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
variable_names = beers.columns.tolist()

for i, var in enumerate(variable_names):
    x, y = correlations[i, 0], correlations[i, 1]
    ax.annotate(
        '',
        xy=(x, y),
        xytext=(0, 0),
        arrowprops=dict(arrowstyle='->', color=colors[i], lw=2)
    )
    ax.text(
        x * 1.08, y * 1.08,
        var,
        fontsize=11,
        color=colors[i],
        ha='center',
        va='center',
        fontweight='bold'
    )

ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100
ax.set_xlabel(f'PC1 ({var1:.1f}%)', fontsize=11)
ax.set_ylabel(f'PC2 ({var2:.1f}%)', fontsize=11)
ax.set_title('Correlation circle (variable space)', fontsize=13)
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Correlation matrix (variable vs PC):')
corr_df = pd.DataFrame(
    correlations[:, :2],
    index=variable_names,
    columns=['PC1', 'PC2']
)
print(corr_df.round(3))

**Answer Q2 — Three groups of variables:**

Looking at the correlation circle:
- **Group 1 (right side, near PC1 axis):** `Taste` and `DgAlcohol` point in the same direction → they are **positively correlated** with each other.
- **Group 2 (right side, pointing down or at angle):** `Bitter` — correlated with Taste/DgAlcohol along PC1 but with a different PC2 component.
- **Group 3 (upper area):** `Thirst` — largely independent of the other variables (points in a different direction).

Variables pointing in **similar directions** are positively correlated; variables pointing in **opposite directions** are negatively correlated; variables with **roughly orthogonal** arrows are uncorrelated.

## Question 3: Was `scale=TRUE` or `scale=FALSE`?

In [ ]:
print('Variable standard deviations (original data):')
print(beers.std().round(4))
print()
print('Variable means:')
print(beers.mean().round(4))

**Answer Q3:** `scale=TRUE` (standardisation before PCA).

The four variables have different scales and ranges (e.g. `DgAlcohol` has a much wider spread than `Bitter`). If `scale=FALSE` were used, PCA would be dominated by the variable with the highest variance, producing a first component that merely captures the unit of measurement rather than meaningful structure. By setting `scale=TRUE`, each variable is standardised to unit variance so that all variables contribute equally to the analysis regardless of their original scale.

## Question 4: Are two principal components sufficient?

In [ ]:
evr = pca.explained_variance_ratio_
cumulative = np.cumsum(evr)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Scree plot
axes[0].bar(range(1, 5), evr * 100, color='steelblue', edgecolor='white')
axes[0].axhline(100 / 4, color='red', linestyle='--', label='Average (25%)')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained variance (%)')
axes[0].set_title('Scree plot')
axes[0].legend()
axes[0].set_xticks(range(1, 5))

# Cumulative
axes[1].plot(range(1, 5), cumulative * 100, 'o-', color='darkorange')
axes[1].axhline(80, color='red', linestyle='--', label='80% threshold')
axes[1].set_xlabel('Number of components')
axes[1].set_ylabel('Cumulative variance (%)')
axes[1].set_title('Cumulative explained variance')
axes[1].legend()
axes[1].set_xticks(range(1, 5))

plt.tight_layout()
plt.show()

print('Individual explained variance:')
for i, (v, c) in enumerate(zip(evr, cumulative), 1):
    print(f'  PC{i}: {v*100:.1f}%  (cumulative: {c*100:.1f}%)')

**Answer Q4:**

- PC1 and PC2 together explain roughly **~75–80%** of total variance.
- The scree plot shows a clear elbow after PC2 — the remaining components contribute much less.
- **Two components are borderline sufficient.** They capture the main structure but miss ~20–25% of the variance. If a third component exceeds the "average" threshold (1/4 = 25%), a third component might be warranted for a more complete picture. In practice, with only 4 variables, 2 components are generally accepted as a reasonable summary.

## Question 5: Interpretation of the four quadrants

*(Refer back to the individuals plot from Question 1)*

From the correlation circle (Q2), the axes can be interpreted as:
- **PC1 (horizontal):** High values → high `Taste`, high `DgAlcohol`, somewhat high `Bitter`. Low values → opposite.
- **PC2 (vertical):** High values → high `Thirst`. Low values → low `Thirst`.

**Quadrant interpretation:**

| Quadrant | PC1 | PC2 | Beer properties |
|----------|-----|-----|------------------|
| Upper-right | + | + | Good taste, high alcohol, thirst-quenching |
| Upper-left | − | + | Milder taste, lower alcohol, thirst-quenching |
| Lower-right | + | − | Good taste, high alcohol, NOT thirst-quenching |
| Lower-left | − | − | Mild taste, low alcohol, NOT thirst-quenching |

**Scenario: drink 50cl, quench thirst, minimise alcohol risk while driving**  
You want high `Thirst` (PC2 > 0) and low `DgAlcohol`. Avoid beers in the **upper-right** quadrant which combine high PC2 (thirst-quenching) with high PC1 (high alcohol). The beer to **avoid** is the one farthest upper-right — e.g. *Red Hooter* (thirst-quenching but check alcohol) or beers with high PC1 + high PC2.

**To avoid bitterness:** Choose beers with low `Bitter` correlation — those at the left side of the correlation circle. Look for beers with low PC1 (lower-left or upper-left quadrant). Beers like *Mousse Blonde* or *Monty Python Holy Grail* (low PC1) tend to be less bitter. **Potential drawback:** low PC1 also correlates with lower `Taste` scores — these may be rated less pleasant by experts.

**To avoid both bitterness and bad taste:** This is contradictory since `Taste` and `Bitter` are positively correlated (both point rightward in the correlation circle). A compromise would be beers with moderate PC1 — not the most bitter but not the worst taste either.